In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import os
import shutil
import pandas as pd
import requests

# Load TDR list
df = pd.read_excel("MP_pipe_data.xlsx")
tdr_list = df["TDR"].astype(str).tolist()

# Download directory
base_download_dir = os.path.join(os.getcwd(), "BOQ_Downloads")
os.makedirs(base_download_dir, exist_ok=True)

# Chrome options
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": base_download_dir,
    "download.prompt_for_download": False,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--remote-debugging-port=9222")
options.add_argument("--disable-gpu")
options.add_argument("--disable-infobars")
options.add_argument("--disable-extensions")

# Initialize WebDriver
driver = webdriver.Chrome(options=options)

# Login
driver.get("https://www.tenderdetail.com/Account/LogOn")
input("🔐 Log in manually, then press ENTER to continue...")

# Summary list
summary = []

# TDR Loop
for tdr in tdr_list:
    print(f"\n🔎 TDR: {tdr}")
    tdr_dir = os.path.join(base_download_dir, tdr)
    os.makedirs(tdr_dir, exist_ok=True)

    # Access search page
    try:
        search_url = f"https://www.tenderdetail.com/registeruser/searchtenders?ServiceType=1&nm={tdr}"
        driver.get(search_url)
        time.sleep(2)
    except Exception as e:
        print(f"❌ Network error for TDR {tdr}: {e}")
        summary.append({"TDR": tdr, "BOQ Files": "Connection Failed", "NIT Files": ""})
        continue

    # Confirm TDR presence
    try:
        tender_brief = driver.find_element(By.CSS_SELECTOR, "#tenderbreif > span").text
        if tdr not in tender_brief:
            print("⚠️ TDR not matched on page")
            summary.append({"TDR": tdr, "BOQ Files": "Not Found", "NIT Files": ""})
            continue
    except:
        summary.append({"TDR": tdr, "BOQ Files": "Not Found", "NIT Files": ""})
        continue

    # Open Notice
    try:
        view_notice = driver.find_element(By.CSS_SELECTOR, ".viewnotice")
        driver.execute_script("arguments[0].click();", view_notice)
        time.sleep(2)
    except:
        print("⚠️ View Notice not clickable")
        summary.append({"TDR": tdr, "BOQ Files": "", "NIT Files": "Not Found"})
        continue

    # Switch tab
    driver.switch_to.window(driver.window_handles[-1])
    time.sleep(2)

    boq_files = []
    nit_files = []

    try:
        all_links = driver.find_elements(By.TAG_NAME, "a")
        for link in all_links:
            href = link.get_attribute("href")
            if not href:
                continue

            filename = href.split("FileName=")[-1] if "FileName=" in href else os.path.basename(href.split("?")[0])

            # BOQ download via Selenium
            if filename.lower().endswith(".xls") and "boq" in filename.lower():
                print(f"📥 BOQ: {filename}")
                try:
                    driver.get(href)
                    boq_files.append(filename)
                    time.sleep(2)
                except Exception as e:
                    print(f"⚠️ BOQ download failed: {e}")

            # NIT download via Requests
            elif filename.lower().endswith(".pdf") and "nit" in filename.lower():
                print(f"📥 NIT: {filename}")
                try:
                    r = requests.get(href)
                    if r.ok:
                        with open(os.path.join(tdr_dir, filename), "wb") as f:
                            f.write(r.content)
                        nit_files.append(filename)
                    else:
                        print(f"⚠️ Failed NIT request: {filename}")
                except Exception as e:
                    print(f"❌ NIT request error: {e}")
    except Exception as e:
        print(f"⚠️ Link parsing error: {e}")

    # Move BOQ files from download dir to tdr dir
    try:
        for f in os.listdir(base_download_dir):
            if f.endswith(".xls") or f.endswith(".xlsx"):
                src = os.path.join(base_download_dir, f)
                dst = os.path.join(tdr_dir, f)
                shutil.move(src, dst)
    except Exception as e:
        print(f"⚠️ File move error: {e}")

    # Switch back to main tab
    if len(driver.window_handles) > 1:
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
    time.sleep(1)

    # Add to summary
    summary.append({
        "TDR": tdr,
        "BOQ Files": ", ".join(boq_files),
        "NIT Files": ", ".join(nit_files)
    })

# Save summary
summary_df = pd.DataFrame(summary)
summary_df.to_excel(os.path.join(base_download_dir, "summary.xlsx"), index=False)

print("\n✅ Done! Summary saved as 'summary.xlsx'")
driver.quit()
